# 2. Attention analysis

Extract the decoder cross-attention of a compound-protein pair and turn it into a
residue-level profile. This is the analysis behind Figure 4 of the paper, shown here
for ABL1 and imatinib, with the non-binding pair MAP2K1-imatinib as a negative control.

**Runtime** — a few seconds on a laptop CPU.

## 1. Setup

The model runs on the CPU. `cfg.training.device` must be set **before** the model is built,
because the device is read at construction time.

In [ ]:
import os
import sys
import warnings

import pandas as pd
from omegaconf import OmegaConf

warnings.filterwarnings('ignore')

# Run from the repository root so that the relative paths below resolve.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

CHECKPOINT = 'checkpoints/application.pth'
assert os.path.exists(CHECKPOINT), (
    f'{CHECKPOINT} not found. Download the weights from Zenodo first '
    '(see checkpoints/README.md).')

cfg = OmegaConf.load('configs/default.yaml')
cfg.model.type = 'aa3di_gmf'   # must match the checkpoint
cfg.training.device = 'cpu'    # set before build_model/load_model
cfg.training.cuda = False

## 2. Build the model and load the weights

Attention maps need `get_attn_maps`, which `predict_single` does not expose, so the
model is built directly here.

In [ ]:
import torch
from omnibind.model import build_model

model = build_model(cfg)
checkpoint = torch.load(CHECKPOINT, map_location=cfg.training.device)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(cfg.training.device)
model.eval()
print('model loaded, epoch', checkpoint['epoch'])

## 3. Prepare one compound-protein pair

In [ ]:
from rdkit import Chem

from omnibind.featurization import (
    mol_features_from_smiles, encode_aa_sequence, encode_3di_sequence,
)
from omnibind.predict import _pack_single

IMATINIB = 'CN1CCN(Cc2ccc(cc2)C(=O)Nc2ccc(C)c(Nc3nccc(n3)-c3cccnc3)c2)CC1'

proteins = pd.read_csv('data/sample/sample_proteins.csv').set_index('gene_name')


def attention_profile(smiles, aa_sequence, sa_sequence):
    """Return per-layer residue attention profiles for one pair."""
    smiles = Chem.MolToSmiles(Chem.MolFromSmiles(smiles), canonical=True)
    atom_feature, adj = mol_features_from_smiles(smiles)
    aa = encode_aa_sequence(aa_sequence)
    sa = encode_3di_sequence(sa_sequence)
    packed = _pack_single([atom_feature], [adj], [aa], [sa])
    compounds, adjs, aas, sas, atom_num, aa_num, sa_num = packed

    device = cfg.training.device
    with torch.no_grad():
        *_, maps = model.get_attn_maps(
            compounds.to(device), adjs.to(device), aas.to(device), sas.to(device),
            atom_num, aa_num, sa_num)
    # Each map is [batch, compound tokens, residues]; row 0 is the virtual atom
    # whose vector feeds the prediction heads.
    return [m.cpu().numpy()[0, 0, :] for m in maps]

## 4. ABL1 and the negative control

The profiles are computed for the binding pair and for a non-binding pair. Each profile
has one value per residue and sums to 1.

In [ ]:
abl1 = proteins.loc['ABL1']
map2k1 = proteins.loc['MAP2K1']

abl1_profiles = attention_profile(IMATINIB, abl1['aa_sequence'], abl1['3di_sequence'])
map2k1_profiles = attention_profile(IMATINIB, map2k1['aa_sequence'], map2k1['3di_sequence'])

print('decoder layers :', len(abl1_profiles))
print('ABL1 residues  :', len(abl1_profiles[0]))
print('MAP2K1 residues:', len(map2k1_profiles[0]))

## 5. Which residues does the model attend to?

Layer 4 (index 3) carries the sharpest signal in the paper, so it is plotted here.
`ABL1_OFFSET` converts the position in this full-length sequence to the conventional
ABL1 numbering used in the literature (for example the gatekeeper T315).

In [ ]:
import numpy as np

LAYER = 3            # 0-based; layer 4 in the paper
ABL1_OFFSET = 0      # this sequence is full-length ABL1 (UniProt P00519)

profile = abl1_profiles[LAYER]
order = np.argsort(profile)[::-1][:10]

pd.DataFrame({
    'position': order + 1 + ABL1_OFFSET,
    'residue': [abl1['aa_sequence'][i] for i in order],
    '3di': [abl1['3di_sequence'][i] for i in order],
    'attention': profile[order],
})

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=False)
for ax, prof, name in zip(axes, [abl1_profiles[LAYER], map2k1_profiles[LAYER]],
                          ['ABL1 (binding)', 'MAP2K1 (non-binding)']):
    ax.plot(np.arange(1, len(prof) + 1), prof, lw=0.8, color='tab:red')
    ax.set_title(f'{name} - imatinib, decoder layer {LAYER + 1}', fontsize=10)
    ax.set_ylabel('attention')
axes[-1].set_xlabel('residue position')
fig.tight_layout()

## 6. Attention across the five decoder layers

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(range(1, 6), [p.max() for p in abl1_profiles], 'o-', label='ABL1')
ax.plot(range(1, 6), [p.max() for p in map2k1_profiles], 's--', label='MAP2K1')
ax.set_xlabel('decoder layer')
ax.set_ylabel('peak attention')
ax.set_xticks(range(1, 6))
ax.legend()
fig.tight_layout()

## 7. Notes

- The profile above uses the virtual-atom row, the token the prediction heads read.
  Averaging over the real atoms (`m[0, 1:, :].mean(axis=0)`) is an alternative.
- Position `i` of a profile corresponds to position `i` of both input sequences; the
  `<cls>`/`<sep>` tokens are removed inside the model.

The same extraction is available as a script, which saves the raw per-layer maps:

```bash
python scripts/attention_map.py \
    test.checkpoint_path=./checkpoints/application.pth \
    test.smiles="<compound_smiles>" \
    test.aa="<aa_sequence>" \
    test.sa="<3di_sequence>" \
    test.name="my_analysis"
```